# Prova sessione unica (come la sessione MATLAB)

Stesse chiamate di `NP_USB_connect_velocity.m` e `trigger_scan.m`, ma divise in celle e nello stesso kernel Python, che resta vivo tra una cella e l'altra (come il workspace di MATLAB).

Esegui le celle in ordine. Il punto della prova: dopo la cella dello scan, rilancia la cella delle query e vedi se risponde ancora `esito=0` o torna `-1`.

Se ti serve ripartire da zero, riavvia il kernel (Restart) e, solo se il laser resta bloccato, riattacca l'USB.

In [1]:
# Cella 1 - carica la DLL (come NET.addAssembly([pwd '\UsbDllWrap.dll']))
import os
import sys
import time

USBADDR = 1
DEVICE_ID = 0x100A  # hex2dec('100A')

dll_path = os.path.join(os.getcwd(), "UsbDllWrap.dll")
if not os.path.isfile(dll_path):
    raise RuntimeError(f"UsbDllWrap.dll non trovata in {dll_path} (il notebook va aperto da scripts/laser)")
print(f"DLL trovata: {dll_path}")
sys.path.append(os.path.dirname(dll_path))

import clr
clr.AddReference("UsbDllWrap")
import Newport
from System.Text import StringBuilder

NP_USB = Newport.USBComm.USB()

DLL trovata: c:\Users\Labo\Desktop\Tesi\scripts\laser\UsbDllWrap.dll


In [2]:
# Cella 2 - apre il dispositivo (una sola volta per sessione)
esito = NP_USB.OpenDevices(DEVICE_ID)
print(f"OpenDevices({hex(DEVICE_ID)}) -> {esito}")
querydata = StringBuilder(64)

OpenDevices(0x100a) -> True


In [5]:
# Cella 3 - query di stato (rilanciabile quante volte vuoi)
esito = NP_USB.Query(USBADDR, "*IDN?", querydata)
print(f"Query '*IDN?' -> esito={esito}, risposta={querydata.ToString()!r}")

esito = NP_USB.Query(USBADDR, "OUTP:STAT?", querydata)
print(f"Query 'OUTP:STAT?' -> esito={esito}, risposta={querydata.ToString()!r}")

Query '*IDN?' -> esito=0, risposta='OK'
Query 'OUTP:STAT?' -> esito=0, risposta='New_Focus 6700 v2.4 03/19/14 SN23163'


In [7]:
# Cella 4 - avvio scan, come trigger_scan.m (una sola volta)
esito = NP_USB.Write(USBADDR, "OUTP:SCAN:START")
print(f"Write 'OUTP:SCAN:START' -> esito={esito}")

Write 'OUTP:SCAN:START' -> esito=0


Ora aspetta che lo scan finisca sul laser, poi rilancia la **Cella 3** (senza riaprire il dispositivo) e annota se risponde `esito=0` o `-1`.